In [ ]:
import streamlit as st
import cv2
import numpy as np
import pandas as pd
import tempfile
import os

from pathlib import Path
from inference_sdk import InferenceHTTPClient, InferenceConfiguration

st.set_page_config(
    page_title="Food & Simpsons Recognizer",
    layout="wide"
)

API_URL = "https://serverless.roboflow.com"
FOOD_MODEL = "food-detection-dataset/1"
SIMPSON_MODEL = "familia-simpsons/1"

@st.cache_resource
def create_client(api_key):
    return InferenceHTTPClient(
        api_url=API_URL,
        api_key=api_key
    ).configure(
        InferenceConfiguration(
            api_key_transport="header"
        )
    )

def draw_rounded_rectangle(
    image,
    x,
    y,
    w,
    h,
    color,
    thickness=3,
    radius=12
):

    x2 = x + w
    y2 = y + h
    radius = min(
        radius,
        w // 2,
        h // 2
    )
    cv2.line(
        image,
        (x + radius, y),
        (x2 - radius, y),
        color,
        thickness,
        cv2.LINE_AA
    )
    cv2.line(
        image,
        (x + radius, y2),
        (x2 - radius, y2),
        color,
        thickness,
        cv2.LINE_AA
    )
    cv2.line(
        image,
        (x, y + radius),
        (x, y2 - radius),
        color,
        thickness,
        cv2.LINE_AA
    )
    cv2.line(
        image,
        (x2, y + radius),
        (x2, y2 - radius),
        color,
        thickness,
        cv2.LINE_AA
    )
    corners = [
        ((x + radius, y + radius), 180),
        ((x2 - radius, y + radius), 270),
        ((x + radius, y2 - radius), 90),
        ((x2 - radius, y2 - radius), 0)
    ]
    for center, angle in corners:
        cv2.ellipse(
            image,
            center,
            (radius, radius),
            angle,
            0,
            90,
            color,
            thickness,
            cv2.LINE_AA
        )

def draw_label(image, label, x, y, color):

    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.65
    thickness = 2

    (text_width, text_height), baseline = (
        cv2.getTextSize(
            label,
            font,
            scale,
            thickness
        )
    )

    label_y = max(y - 8, text_height + 15)
    cv2.rectangle(
        image,
        (x, label_y - text_height - 12),
        (
            x + text_width + 12,
            label_y + baseline
        ),
        color,
        -1
    )

    cv2.putText(
        image,
        label,
        (x + 6, label_y - 5),
        font,
        scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA
    )

def get_predictions(client, image, model_id):

    with tempfile.NamedTemporaryFile(
        suffix=".jpg",
        delete=False
    ) as temp_file:
        image_path = temp_file.name

    try:
        success = cv2.imwrite(
            image_path,
            image
        )
        if not success:
            raise RuntimeError("Could not encode image.")
        result = client.infer(
            image_path,
            model_id=model_id
        )
        return result.get("predictions", [])

    finally:
        if os.path.exists(image_path):
            os.remove(image_path)

def annotate_image(
    image,
    predictions,
    confidence_threshold
):

    output = image.copy()
    detections = []
    height, width = image.shape[:2]
    for prediction in predictions:

        confidence = float(
            prediction.get("confidence", 0)
        )
        if confidence < confidence_threshold:
            continue
        class_name = prediction.get(
            "class",
            "None"
        )
        center_x = float(prediction["x"])
        center_y = float(prediction["y"])

        box_width = float(prediction["width"])
        box_height = float(prediction["height"])

        x1 = int(center_x - box_width / 2)
        y1 = int(center_y - box_height / 2)

        x2 = int(center_x + box_width / 2)
        y2 = int(center_y + box_height / 2)

        x1 = max(0, min(x1, width - 1))
        y1 = max(0, min(y1, height - 1))

        x2 = max(0, min(x2, width - 1))
        y2 = max(0, min(y2, height - 1))

        if x2 <= x1 or y2 <= y1:
            continue

        color = (60, 190, 110)
        draw_rounded_rectangle(
            output,
            x1,
            y1,
            x2 - x1,
            y2 - y1,
            color
        )
        label = (
            f"{class_name} "
            f"{confidence * 100:.1f} %"
        )

        draw_label(
            output,
            label,
            x1,
            y1,
            color
        )

        detections.append({
            "Class": class_name,
            "Confidence": round(
                confidence * 100,
                2
            ),
            "X": x1,
            "Y": y1,
            "Width": x2 - x1,
            "Height": y2 - y1
        })

    return output, detections

def process_image(
    image,
    client,
    model_id,
    confidence_threshold
):
    predictions = get_predictions(
        client,
        image,
        model_id
    )
    output, detections = annotate_image(
        image,
        predictions,
        confidence_threshold
    )
    return output, detections

def process_video(
    uploaded_file,
    client,
    model_id,
    confidence_threshold,
    frame_interval
):

    input_file = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp4"
    )
    input_file.write(
        uploaded_file.getvalue()
    )
    input_file.close()
    output_file = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp4"
    )
    output_file.close()
    cap = cv2.VideoCapture(
        input_file.name
    )

    if not cap.isOpened():

        os.remove(input_file.name)
        raise RuntimeError(
            "Failed to Open Uploaded Video"
        )

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        fps = 25

    width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )
    height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )
    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )
    writer = cv2.VideoWriter(
        output_file.name,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    frame_number = 0
    last_predictions = []
    progress = st.progress(0)
    status = st.empty()

    try:
        while True:
            success, frame = cap.read()
            if not success:
                break

            if frame_number % frame_interval == 0:
                last_predictions = get_predictions(
                    client,
                    frame,
                    model_id
                )
                status.write(
                    f"Online inference : "
                    f"Frame {frame_number}"
                )

            annotated_frame, _ = annotate_image(
                frame,
                last_predictions,
                confidence_threshold
            )
            annotated_frame = cv2.cvtColor(
                annotated_frame,
                cv2.COLOR_RGB2BGR
            )
            writer.write(
                annotated_frame
            )

            frame_number += 1
            if total_frames > 0:

                progress.progress(
                    min(
                        frame_number / total_frames,
                        1.0
                    )
                )

    finally:
        cap.release()
        writer.release()

        progress.empty()
        status.empty()

        os.remove(input_file.name)

    return output_file.name

def display_results(image, output, detections):

    original, processed = st.columns(2)
    with original:
        st.subheader("Original Image")

        st.image(
            cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB
            ),
            use_container_width=True
        )
    with processed:

        st.subheader("Detection Results")
        st.image(
            cv2.cvtColor(
                output,
                cv2.COLOR_BGR2RGB
            ),
            use_container_width=True
        )
    st.divider()

    col1, col2 = st.columns(2)
    with col1:

        st.metric(
            "Objects Detected",
            len(detections)
        )

    with col2:
        average_confidence = (
            np.mean([
                item["Confidence"]
                for item in detections
            ])
            if detections else 0
        )
        st.metric(
            "Average Confidence",
            f"{average_confidence:.2f} %"
        )

    if detections:
        st.subheader("Detection Information")
        st.dataframe(
            pd.DataFrame(detections),
            use_container_width=True,
            hide_index=True
        )
    else:

        st.info(
            "Failed to Detect any Objects "
        )

st.title("Food & Simpsons Vision AI")

with st.expander("API Configuration", expanded=True):

    api_key = st.text_input(
        "Roboflow Api Key",
        type="password",
        value=os.getenv("ROBOFLOW_API_KEY", ""),
    )

if not api_key:
    st.warning(
        "Enter your Robo Flow Api Key to Activate both Models"
    )
    st.stop()
try:
    client = create_client(api_key)
except Exception as error:
    st.error(f"Api Connection Error : {error}")
    st.stop()

food_tab, simpson_tab = st.tabs([
    "Food Recognizer",
    "Simpson Character Recognizer"
])

with food_tab:

    st.header("Food Recognizer")
    confidence = st.slider(
        "Confidence Threshold",
        min_value=0.05,
        max_value=0.95,
        value=0.35,
        step=0.05,
        key="food_confidence"
    )
    uploaded_file = st.file_uploader(
        "Upload an Image or Video",
        type=["jpg", "jpeg", "png", "mp4"],
        key="food_upload"
    )
    if uploaded_file is not None:
        is_video = uploaded_file.name.lower().endswith(
            ".mp4"
        )
        if is_video:
            frame_interval = st.slider(
                "Run Inference Every N Frames",
                min_value=1,
                max_value=30,
                value=5,
                key="food_interval"
            )
            if st.button(
                "Recognize Food",
                key="food_video_button"
            ):
                try:
                    with st.spinner(
                        "Processing Video"
                    ):
                        output_path = process_video(
                            uploaded_file,
                            client,
                            FOOD_MODEL,
                            confidence,
                            frame_interval
                        )

                    st.success(
                        "Food Recognition Completed!"
                    )
                    st.video(output_path)
                except Exception as error:
                    st.error(
                        f"Video Processing Failed : {error}"
                    )
        else:
            file_bytes = np.frombuffer(
                uploaded_file.getvalue(),
                np.uint8
            )
            image = cv2.imdecode(
                file_bytes,
                cv2.IMREAD_COLOR
            )
            if image is None:

                st.error("Invalid Image File")

            elif st.button(
                "Recognize Food",
                key="food_image_button"
            ):
                try:
                    with st.spinner(
                        "Processing Image"
                    ):
                        output, detections = process_image(
                            image,
                            client,
                            FOOD_MODEL,
                            confidence
                        )
                    display_results(
                        image,
                        output,
                        detections
                    )
                except Exception as error:

                    st.error(
                        f"Inference Failed : {error}"
                    )

with simpson_tab:

    st.header("Simpson Character Recognizer")
    confidence = st.slider(
        "Confidence Threshold",
        min_value=0.05,
        max_value=0.95,
        value=0.35,
        step=0.05,
        key="simpson_confidence"
    )
    uploaded_file = st.file_uploader(
        "Upload an Image or Video",
        type=["jpg", "jpeg", "png", "mp4"],
        key="simpson_upload"
    )
    if uploaded_file is not None:

        is_video = uploaded_file.name.lower().endswith(
            ".mp4"
        )
        if is_video:

            frame_interval = st.slider(
                "Run Inference Every N Frames",
                min_value=1,
                max_value=30,
                value=5,
                key="simpson_interval"
            )

            if st.button(
                "Recognize Characters",
                key="simpson_video_button"
            ):
                try:
                    with st.spinner(
                        "Processing Video"
                    ):
                        output_path = process_video(
                            uploaded_file,
                            client,
                            SIMPSON_MODEL,
                            confidence,
                            frame_interval
                        )

                    st.success(
                        "Character Recognition Completed!"
                    )
                    st.video(output_path)
                except Exception as error:
                    st.error(
                        f"Video Processing Failed : {error}"
                    )
        else:
            file_bytes = np.frombuffer(
                uploaded_file.getvalue(),
                np.uint8
            )
            image = cv2.imdecode(
                file_bytes,
                cv2.IMREAD_COLOR
            )
            if image is None:
                st.error("Invalid Image File")
            elif st.button(
                "Recognize Characters",
                key="simpson_image_button"
            ):
                try:
                    with st.spinner(
                        "Processing Image"
                    ):
                        output, detections = process_image(
                            image,
                            client,
                            SIMPSON_MODEL,
                            confidence
                        )
                    display_results(
                        image,
                        output,
                        detections
                    )
                except Exception as error:
                    st.error(
                        f"Inference Failed : {error}"
                    )